#Oracle

This code implements a padding oracle. The only function you need to use is `Oracle.valid_padding`, and (if you want to for performance reasons), `Oracle.valid_padding_multiple` (see example).

In [39]:
import random
import socket
from contextlib import AbstractContextManager

# Number of bytes for every AES block.
BLOCK_LENGTH = 16


class Oracle(AbstractContextManager):
    def __init__(self, ip: str, port: int):
        self.s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.ip = ip
        self.port = port

    def __enter__(self):
        self.s.connect((self.ip, self.port))
        print(f"Connected to {self.ip}:{self.port} successfully.")

        return self

    def __exit__(self, __exc_type, __exc_value, __traceback):
        super().__exit__(__exc_type, __exc_value, __traceback)
        self.s.send(b"\x00")
        self.s.close()

    # Packet Structure: < num_blocks(1) || ciphertext(16*num_blocks) || null-terminator(1) >
    def send_ciphertext(self, c: bytes):
        num_blocks, rest = divmod(len(c), BLOCK_LENGTH)

        if rest != 0:
            raise ValueError(
                f"Number of bytes ({len(c)}) must be a multiple of BLOCK_LENGTH"
            )

        msg = bytes([num_blocks]) + c + b"\x00"
        self.s.send(msg)

    # Response: <0/1> (ASCII) || 0x00
    def recv_response(self) -> bool:
        resp = self.s.recv(2)

        if len(resp) != 2 or resp[1] != 0:
            raise ValueError("Invalid response from server")

        return bool(int(resp[:1]))

    @staticmethod
    def normalize(c: bytes | list[int]) -> bytes:
        if isinstance(c, list):
            c = bytearray(c)

        return c

    def valid_padding(self, c: bytes | list[int]) -> bool:
        """Returns whether Dec_K(c) has valid padding."""
        c = self.normalize(c)

        self.send_ciphertext(c)
        return self.recv_response()

    def valid_padding_multiple(self, cs: list[bytes | list[int]]) -> list[bool]:
        """
        Check for a list of ciphertexts c, whether Dec_k(c[i]) has valid padding.

        When you know multiple ciphertexts in advance that you want to check,
        this function can be much faster than multiple calls to
        Oracle.valid_padding.
        """
        for c in cs:
            self.send_ciphertext(self.normalize(c))

        return [self.recv_response() for _ in range(len(cs))]


# Unpadding

In [40]:
def unpad(b: bytes) -> bytes:
    padding_length = b[-1]

    if not all(x == 0 for x in b[-padding_length: -1]):
        raise ValueError("Invalid padding")

    return b[:-padding_length]

#Solution

Start with decrypting the ciphertext from c1 which is 32 bytes long, 2 blocks of 16 bytes. The plaintext is plain English.

Once your padding-oracle attack program works, you can try to decrypt the other (longer) ciphertexts as well. They are encrypted under the same key. Some of them contain an emoji 😃.

In [ ]:
c1 = ('b6fe3cd947416682eb979ece392ebba1'
      'f2b5a00d8d6d0b96889da94f7dfebfe2')
c2 = '22465c312c0590bd7e04e06494babac2b4dc8eb7f87e7da935a3ead1917ee1eff3d4917e18f057927116e69d641a42c8'
c3 = '51387161431cbe95d179a33b93d9a80ce89dbebfdad2adb8e4d405116a0cd6f6'
c4 = 'bde3ac7790243aa158a326d08b92883e853f8ebb6d73ed0df372a2c6a340e522646657b99833e3b7148f0a9a4bd1982d'
c5 = '71e563f9c5c5d89c5cf4d50ee5141b458d49b584d29e7d525a714a5295e3eb40c3a9f231c3fd7945e2a3024b3d04c9a0407d46ebd59c3834b02ce824047a9867d34aba5047f570346077dd53f3e669c9'
c6 = '8fb27315cee1589c0d88d3f11c6aba3be391b0d19c0ba485bb77ff456c82fcd6afa6c1ea19c47a40408b695477ee2cd9da53d8bb90a8e3232608e30d114fe2f773faaa020c73fc77c186bcf74842c487'


def decrypt(o: Oracle, c: bytes) -> bytes:
    """Using access to the padding oracle, find the decryption of c."""

    # Python has two ways of representing a string of raw bytes:
    #
    # - One option is to use the bytes type, which is similar to the the
    #   familiar str type.
    # - You can also represent raw bytes as a list of integer with values in
    #   {0, 1, ..., 255}, which represent the numerical value of the every byte.
    #
    # Depending on what you are trying to do, both representations can be very
    # useful!
    #
    # The oracle accepts both representations, whichever is more convenient for
    # you. You can also convert between bytes and list[int]

    c_as_list: list[int] = [i for i in c]
    assert bytes(c_as_list) == c

    assert o.valid_padding(c)
    assert o.valid_padding(c_as_list)

    cs = [bytes.fromhex(c) for c in (c1, c2, c3, c4, c5, c6)]
    assert o.valid_padding_multiple(cs + [bytes(32)]) == [True] * 6 + [False]

    block_size = 16
    assert len(c) % block_size == 0

    blocks = [c[i:i + block_size] for i in range(0, len(c), block_size)]
    decrypted = []

    for block in range(1, len(blocks)):
        original_vector = blocks[block - 1]
        current_vector = bytearray(random.randbytes(block_size))

        current_intermediate = [0] * block_size
        decrypted_block = [0] * block_size

        for byte in range(block_size - 1, -1, -1):
            padding_value = block_size - byte

            if byte < block_size - 1:
                current_vector[-1] = current_intermediate[-1] ^ padding_value

            def create_guess(guess):
                array = bytearray(current_vector)
                array[byte] = guess
                return array + blocks[block]

            guesses = [create_guess(i) for i in range(256)]
            guesses = o.valid_padding_multiple(guesses)

            for guess in range(256):
                if not guesses[guess]:
                    continue

                current_intermediate[byte] = current_vector[byte] = guess
                decrypted_block[byte] = guess ^ original_vector[byte]

                if byte == block_size - 1:
                    current_intermediate[byte] ^= 1
                    decrypted_block[byte] = current_intermediate[byte] ^ original_vector[byte]

                if byte == block_size - 1 and byte == original_vector[-1] and current_vector[-2] == original_vector[-2]:
                    # Possible false positive?, continue to check if there are other valid options
                    print("continuing to check for other valid options")
                    continue

                break

        decrypted += decrypted_block

    return o.normalize(decrypted)


# Use a context manager to automatically close the connection to the server.
with Oracle("178.62.241.83", 8080) as oracle:
    # decrypt() returns the padded message as raw bytes

    for c in (c1, c2, c3, c4, c5, c6):
        m_bytes = decrypt(oracle, bytes.fromhex(c))

        # To print the message properly, you need to remove the padding, and decode
        # the UTF-8 bytes, to the proper characters. We have provided the unpadding
        # function for you.
        m_str = unpad(m_bytes).decode()
        print("Decrypted message:", m_str)


Connected to 178.62.241.83:8080 successfully.
Decrypted message: A message!
Decrypted message: There can be multiple blocks
